# Mi plan de trabajo Silver

Este notebook consulta cierres Bronze, auditoría Silver y destinos del mismo snapshot.
No transforma ni publica datos analíticos.
Primero ejecuto la vista previa con emit_task_values=false.
El Job usa emit_task_values=true para entregar las listas a sus diez ciclos.
Una lista vacía significa que esa entidad no tiene trabajo dentro del plan.
La política de service_dates está en databricks/config/silver_planner.json.


In [0]:
# BLOQUE 1 — Repositorio y parámetros del plan
from pathlib import Path
import json
import sys

defaults = {
    "repo_root": "", "catalog": "workspace", "schema": "gtfs_bogota",
    "feed_version_from": "", "feed_version_to": "", "max_snapshots": "20",
    "max_metadata_rows": "5000", "emit_task_values": "false",
}
for key, default in defaults.items():
    dbutils.widgets.text(key, default)
parameters = {key: dbutils.widgets.get(key).strip() for key in defaults}
root_parameter = parameters.pop("repo_root")
if root_parameter and not Path(root_parameter).is_absolute():
    raise ValueError("repo_root debe ser una ruta absoluta de Workspace.")
current = Path.cwd().resolve()
roots = [Path(root_parameter)] if root_parameter else [current, *current.parents]
project_root = next((p for p in roots if (p / "databricks/gtfs_silver/planner.py").is_file()), None)
if project_root is None:
    raise RuntimeError("Sincroniza el repositorio completo y configura repo_root.")
module_path = str(project_root / "databricks")
if module_path not in sys.path:
    sys.path.insert(0, module_path)
from gtfs_silver.pipeline import load_registry
from gtfs_silver.planner_runtime import create_plan, emit_plan
emit_task_values = parameters.pop("emit_task_values").lower()
if emit_task_values not in {"true", "false"}:
    raise ValueError("emit_task_values debe ser true o false.")
for key in ("max_snapshots", "max_metadata_rows"):
    parameters[key] = int(parameters[key])


In [0]:
# BLOQUE 2 — Plan de lectura y decisiones visibles
plan = create_plan(spark, registry=load_registry(), **parameters)
print(json.dumps({key: plan[key] for key in ("planner_version", "status", "counts", "input_versions")}, ensure_ascii=False, indent=2))
if plan["decisions"]:
    decision_rows = [tuple(str(row.get(key, "")) for key in (
        "entity", "feed_version", "snapshot_sha256", "status", "reason",
    )) for row in plan["decisions"]]
    decision_schema = "entity string, feed_version string, snapshot_sha256 string, status string, reason string"
    display(spark.createDataFrame(decision_rows, decision_schema))
for entity, items in plan["work_items"].items():
    print(f"{entity}: {len(items)} trabajo(s)")


In [0]:
# BLOQUE 3 — Entrega de listas dentro del mismo Job
if emit_task_values == "true":
    emit_plan(dbutils, plan)
    print("Listas entregadas al Job. Este notebook no publicó tablas Silver.")
else:
    print("Vista previa: no entregué task values ni escribí tablas.")
    print(json.dumps(plan["work_items"], ensure_ascii=False, indent=2))
